# Analyse des résultats d'optimisation

Ce notebook parcourt les fichiers JSON dans `results/`, extrait les parts de mélange (`saf_ftg_mandate_share`, `saf_co2_mandate_share`, `lcaf_mandate_share`) et reconstitue la part kérosène implicite (100 % moins la somme). Les résultats sont tracés en aires empilées pour visualiser la contribution de chaque filière.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Charger les fichiers JSON du répertoire courant
results_dir = Path("./")
json_files = sorted(results_dir.glob("*.json"))

# Ajouter le fichier de référence
reference_file = Path("./../../data/ltag_data_outputs/is2medium_outputs.json").resolve()
if reference_file.exists():
    json_files.append(reference_file)
    print(f"Fichier de référence ajouté: {reference_file}")
else:
    print(f"Avertissement: fichier de référence non trouvé: {reference_file}")

json_files

In [ ]:
import re
from matplotlib import cm

BASE_YEAR = 2000
PLOT_START_YEAR = 2020
PLOT_END_YEAR = 2070


def load_vector_outputs(path: Path):
    with path.open() as f:
        data = json.load(f)
    return data.get("vector_outputs", {}), data


def get_cut_year(path: Path):
    """Return cut year if filename contains cut_YYYY, else None."""
    match = re.search(r"cut_(\d{4})", path.name)
    return int(match.group(1)) if match else None


def apply_cut(df: pd.DataFrame, years: np.ndarray, cut_year: int | None):
    if cut_year is None:
        return df, years
    mask = years <= cut_year
    if not mask.any():
        return df.iloc[0:0], years[0:0]
    return df.loc[mask], years[mask]


def extract_shares(vector_outputs: dict):
    """Extract shares from vector outputs."""
    ftg_list = vector_outputs.get("saf_ftg_mandate_share", [])
    co2_list = vector_outputs.get("saf_co2_mandate_share", [])
    lcaf_list = vector_outputs.get("lcaf_mandate_share", [])

    ftg = pd.Series(ftg_list)
    co2 = pd.Series(co2_list)
    lcaf = pd.Series(lcaf_list)

    df = pd.DataFrame(
        {
            "saf_ftg": ftg,
            "saf_co2": co2,
            "lcaf": lcaf,
        }
    )

    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.fillna(method="ffill").fillna(0)
    df["kerosene"] = (100 - df[["saf_ftg", "saf_co2", "lcaf"]].sum(axis=1)).clip(lower=0)

    return df


def plot_blend(df: pd.DataFrame, title: str):
    years = BASE_YEAR + np.arange(len(df))
    x_end = min(PLOT_END_YEAR, years[-1]) if len(years) else PLOT_END_YEAR
    # Utilise les couleurs du colormap 'Set2'
    set2 = cm.get_cmap("Set2")
    colors = [set2(0), set2(5), set2(4), set2(2)]
    fig, ax = plt.subplots(figsize=(9, 5))
    # Inverser l'ordre pour que Kerosene soit en haut et SAF CO2 en bas
    ax.stackplot(
        years,
        [df["saf_co2"], df["saf_ftg"], df["lcaf"], df["kerosene"]],
        labels=["E-Fuel", "Biofuel", "LCAF", "Kerosene"],
        colors=colors,
        alpha=0.85,
    )
    ax.set_ylabel("Blend share [%]")
    ax.set_xlabel("Year")
    ax.set_title(title)

    ax.set_ylim(0, 100)

    ax.set_xlim(PLOT_START_YEAR, x_end)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")
    return fig, ax

In [ ]:
results = []
n_plots = len(json_files)

# Calculer les dimensions de la grille
n_cols = 2
n_rows = (n_plots + n_cols - 1) // n_cols

# Créer la figure avec les subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))

# Assurer que axes est toujours un tableau 2D
if n_plots == 1:
    axes = np.array([[axes]])
elif n_rows == 1:
    axes = axes.reshape(1, -1)
elif n_cols == 1:
    axes = axes.reshape(-1, 1)

# Créer les plots
for idx, path in enumerate(json_files):
    row = idx // n_cols
    col = idx % n_cols
    ax = axes[row, col]

    vec_outputs, raw = load_vector_outputs(path)
    df = extract_shares(vec_outputs)
    years = BASE_YEAR + np.arange(len(df))
    cut_year = get_cut_year(path)
    df, years = apply_cut(df, years, cut_year)

    if len(years) == 0:
        ax.text(0.5, 0.5, "Aucune donnée avant le cut", ha="center", va="center")
        ax.axis("off")
        continue

    title = f"Blend shares – {path.name}"

    # Utiliser les couleurs du colormap 'Set2'
    set2 = cm.get_cmap("Set2")
    colors = [set2(0), set2(5), set2(4), set2(2)]

    ax.stackplot(
        years,
        [df["saf_co2"], df["saf_ftg"], df["lcaf"], df["kerosene"]],
        labels=["E-Fuel", "Biofuel", "LCAF", "Kerosene"],
        colors=colors,
        alpha=0.85,
    )
    ax.set_ylabel("Blend share [%]")
    ax.set_xlabel("Year")
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, 100)
    ax.set_xlim(PLOT_START_YEAR, PLOT_END_YEAR)
    ax.legend(loc="upper left", fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)

    # Zone grisée après le cut, si présent
    if cut_year is not None:
        ax.axvspan(cut_year, PLOT_END_YEAR, color="grey", alpha=0.15)

    results.append({"file": path.name, "data": df, "cut_year": cut_year})

# Masquer les axes inutilisés
for idx in range(n_plots, n_rows * n_cols):
    row = idx // n_cols
    col = idx % n_cols
    axes[row, col].axis("off")

plt.tight_layout()
plt.show()

results

In [ ]:
vec_outputs.get("saf_ftg_mandate_share")

In [ ]:
# Figure supplémentaire: parts de mélange en lignes superposées
n_plots = len(json_files)

# Grille (2 colonnes)
n_cols = 2
n_rows = (n_plots + n_cols - 1) // n_cols

fig_lines, axes_lines = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))

# Normaliser la forme des axes
if n_plots == 1:
    axes_lines = np.array([[axes_lines]])
elif n_rows == 1:
    axes_lines = axes_lines.reshape(1, -1)
elif n_cols == 1:
    axes_lines = axes_lines.reshape(-1, 1)

set2 = cm.get_cmap("Set2")
colors = [set2(0), set2(5), set2(4), set2(2)]
labels = ["E-Fuel", "Biofuel", "LCAF", "Kerosene"]

bottom_left_row = axes_lines.shape[0] - 1
bottom_left_col = 0

for idx, item in enumerate(results):
    row = idx // n_cols
    col = idx % n_cols
    ax = axes_lines[row, col]

    df = item["data"]
    years = BASE_YEAR + np.arange(len(df))
    cut_year = item.get("cut_year")
    if len(years) == 0:
        ax.text(0.5, 0.5, "Aucune donnée avant le cut", ha="center", va="center")
        ax.axis("off")
        continue

    title = f"Blend shares (lines) – {item['file']}"

    # Tracer les lignes superposées (déjà tronquées au cut)
    ax.plot(years, df["saf_co2"], color=colors[0], label=labels[0], linewidth=2)
    ax.plot(years, df["saf_ftg"], color=colors[1], label=labels[1], linewidth=2)
    ax.plot(years, df["lcaf"], color=colors[2], label=labels[2], linewidth=2)
    ax.plot(years, df["kerosene"], color=colors[3], label=labels[3], linewidth=2)

    ax.set_ylabel("Share [%]")
    ax.set_xlabel("Year")
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, 100)
    ax.set_xlim(PLOT_START_YEAR, PLOT_END_YEAR)
    if row == bottom_left_row and col == bottom_left_col:
        ax.legend(loc="upper left", fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)

    # Zone grisée après le cut, si présent
    if cut_year is not None:
        ax.axvspan(cut_year, PLOT_END_YEAR, color="grey", alpha=0.15)

# Masquer les axes inutilisés
for idx in range(n_plots, n_rows * n_cols):
    row = idx // n_cols
    col = idx % n_cols
    axes_lines[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Figure par catégorie de fuel : chaque subplot représente un fuel, avec une ligne par fichier JSON
fuel_categories = ["E-Fuel", "Biofuel", "LCAF", "Kerosene"]
fuel_keys = ["saf_co2", "saf_ftg", "lcaf", "kerosene"]

set2 = plt.cm.get_cmap("Set2")
fuel_colors = {
    "E-Fuel": set2(0),
    "Biofuel": set2(5),
    "LCAF": set2(4),
    "Kerosene": set2(2),
}

# Années de contrôle pour les ticks
control_years = [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060, 2065, 2070]

# Mapping des labels pour cell 10
label_mapping_cell10 = {
    "slsqp_co2_afree_all.json": r"None - Obj: Cum. $CO_2$",
    "slsqp_co2_free.json": r"Ramp-Up (RU) - Obj: Cum. $CO_2$",
    "slsqp_co2_no_degrowth.json": r"RU + No Ramp-Down (NRD) - Obj: Cum. $CO_2$",
    "slsqp_co2_ressource.json": r"RU + Resources Availability (RA) - Obj: Cum. $CO_2$",
}

# Créer une figure avec 4 subplots (1 par catégorie)
fig_by_fuel, axes_by_fuel = plt.subplots(2, 2, figsize=(18, 12))
axes_by_fuel = axes_by_fuel.flatten()

# Filtrer les résultats pour n'inclure que les fichiers demandés (correspondance exacte)
files_to_include = [
    "is2medium_outputs.json",
    "slsqp_co2_afree_all.json",
    "slsqp_co2_free.json",
    "slsqp_co2_no_degrowth.json",
    "slsqp_co2_ressource.json",
]
filtered_results = [item for item in results if item["file"] in files_to_include]

# Générer une palette de couleurs pour les différents fichiers (sauf la référence)
n_files = len(filtered_results)
file_colors = plt.cm.tab10(np.linspace(0, 1, n_files))

for fuel_idx, (fuel_name, fuel_key) in enumerate(zip(fuel_categories, fuel_keys)):
    ax = axes_by_fuel[fuel_idx]

    # Tracer une ligne pour chaque fichier JSON
    for file_idx, item in enumerate(filtered_results):
        df = item["data"]
        years = BASE_YEAR + np.arange(len(df))
        if len(years) == 0:
            continue

        filename = item["file"]

        # Cas spécial pour la référence
        if "is2medium_outputs" in filename:
            label = "IS2 reference"
            color = "#404040"  # Gris foncé
            linewidth = 2.5
            marker = None
            markersize = 0
            markevery = None
            alpha = 1.0
        else:
            label = label_mapping_cell10.get(filename, filename.replace(".json", ""))
            color = file_colors[file_idx]
            linewidth = 2
            marker = "o"
            markersize = 4
            mark_indices = [i for i, year in enumerate(years) if year in control_years]
            markevery = mark_indices if mark_indices else None
            alpha = 0.8

        ax.plot(
            years,
            df[fuel_key],
            color=color,
            label=label,
            linewidth=linewidth,
            alpha=alpha,
            marker=marker,
            markersize=markersize,
            markevery=markevery,
        )

    ax.set_ylabel("Share [%]", fontsize=12)
    ax.set_xlabel("Year", fontsize=12)
    ax.set_title(f"{fuel_name}", fontsize=14, fontweight="bold")
    ax.set_ylim(0, 105)
    ax.set_xlim(PLOT_START_YEAR, PLOT_END_YEAR)
    tick_years = [y for y in control_years if y <= PLOT_END_YEAR]
    ax.set_xticks(tick_years)
    ax.set_xticklabels([str(y) for y in tick_years], rotation=45)
    if fuel_idx == 2:
        ax.legend(loc="upper left", fontsize=15, ncol=1, framealpha=0.9)
    ax.grid(True, alpha=0.3, which="both")
    ax.grid(True, alpha=0.5, which="major", linewidth=0.8)

plt.tight_layout()
plt.savefig("fuel_categories_selected.pdf", bbox_inches="tight")

plt.show()

In [ ]:
# Donut plot : cumulative CO2 emissions (cell 10 scenarios) — segments align from a common start
from pathlib import Path
from matplotlib.patches import Wedge
import matplotlib.pyplot as plt

index_2070 = 2070 - BASE_YEAR
values = []
labels = []
files_cell10 = [
    "is2medium_outputs.json",
    "slsqp_co2_afree_all.json",
    "slsqp_co2_free.json",
    "slsqp_co2_no_degrowth.json",
    "slsqp_co2_ressource.json",
]
filtered_cell10 = [item for item in results if item["file"] in files_cell10]
colors = plt.cm.tab10(
    np.linspace(0, 1, len(filtered_cell10))
)  # utilisé uniquement pour la barre finale


# Helper to resolve file path
def _resolve_path(name: str) -> Path:
    if "is2medium_outputs" in name:
        return reference_file
    candidate = results_dir / name
    return candidate if candidate.exists() else Path(name)


for file_idx, item in enumerate(filtered_cell10):
    fname = item["file"]
    path = _resolve_path(fname)
    if not path.exists():
        print(f"Fichier manquant pour {fname} : {path}")
        continue

    vec_outputs, _ = load_vector_outputs(path)
    cumu = vec_outputs.get("cumulative_co2_emissions", [])
    if not cumu:
        print(f"Pas de 'cumulative_co2_emissions' dans {fname}")
        continue

    value = cumu[index_2070] if len(cumu) > index_2070 else cumu[-1]
    values.append(value)
    labels.append(label_mapping_cell10.get(fname, fname.replace(".json", "")))

if values:
    max_val = 100.0  # 100 GtCO2 correspond au demi-cercle complet
    start_angle = 180  # point de départ (à gauche)
    radius = 0.5
    width = 0.25

    fig_donut, ax_donut = plt.subplots(figsize=(10, 6))

    # Cercles fins intérieur/extérieur - seulement demi-cercle supérieur
    outer_circle = Wedge(
        center=(0, 0),
        r=radius,
        theta1=0,
        theta2=180,
        width=0,
        fill=False,
        edgecolor="#b0b0b0",
        linewidth=1.0,
        alpha=0.8,
    )
    inner_circle = Wedge(
        center=(0, 0),
        r=radius - width,
        theta1=0,
        theta2=180,
        width=0,
        fill=False,
        edgecolor="#b0b0b0",
        linewidth=1.0,
        alpha=0.8,
    )
    ax_donut.add_patch(outer_circle)
    ax_donut.add_patch(inner_circle)

    # Fermer les demi-cercles avec des lignes verticales
    ax_donut.plot([-radius, -(radius - width)], [0, 0], color="#b0b0b0", linewidth=1.0, alpha=0.8)
    ax_donut.plot([radius, radius - width], [0, 0], color="#b0b0b0", linewidth=1.0, alpha=0.8)

    # Fond gris couvrant le demi-cercle entier (100 Gt) - seulement la partie supérieure
    full_wedge = Wedge(
        center=(0, 0),
        r=radius,
        theta1=0,
        theta2=180,
        width=width,
        facecolor="#e6e6e6",
        edgecolor="white",
        linewidth=1.0,
        alpha=0.6,
    )
    ax_donut.add_patch(full_wedge)

    for idx, (val, label, color) in enumerate(zip(values, labels, colors)):
        proportion = min(val, max_val) / max_val
        end_angle = start_angle - proportion * 180  # Utiliser 180 au lieu de 360

        # Utiliser noir pour IS2 reference
        fname = filtered_cell10[idx]["file"]
        if "is2medium_outputs" in fname:
            base_color = np.array([0, 0, 0])  # Noir
        else:
            # Convertir la couleur matplotlib en RGB
            base_color = np.array(color[:3])

        # Barre finale pour marquer la fin
        angle_rad = np.deg2rad(end_angle)
        r_inner = radius - width
        r_outer = radius
        x1, y1 = r_inner * np.cos(angle_rad), r_inner * np.sin(angle_rad)
        x2, y2 = r_outer * np.cos(angle_rad), r_outer * np.sin(angle_rad)
        ax_donut.plot(
            [x1, x2], [y1, y2], color=base_color, linewidth=3.0, solid_capstyle="round", label=label
        )

    # Graduations tous les 10 (sans unité), étiquettes à plat, en dehors du cercle externe
    for tick in range(0, 101, 10):
        frac = tick / max_val
        angle_tick = start_angle - frac * 180  # Utiliser 180 au lieu de 360
        angle_rad = np.deg2rad(angle_tick)
        r_inner = radius - width
        r_outer = radius
        x1, y1 = r_inner * np.cos(angle_rad), r_inner * np.sin(angle_rad)
        x2, y2 = r_outer * np.cos(angle_rad), r_outer * np.sin(angle_rad)
        ax_donut.plot([x1, x2], [y1, y2], color="#b0b0b0", linewidth=0.8, alpha=0.7)

        # Positionner les chiffres à plat, proche du cercle externe
        r_tick_label = r_outer + 0.05
        x_txt = r_tick_label * np.cos(angle_rad)
        y_txt = r_tick_label * np.sin(angle_rad)

        ax_donut.text(
            x_txt,
            y_txt,
            f"{tick}",
            fontsize=20,
            ha="center",
            va="center",
            color="#7D7A7A",
            rotation=0,
        )

    ax_donut.set_aspect("equal")
    ax_donut.set_xlim(-0.75, 0.75)
    ax_donut.set_ylim(0, 0.75)
    ax_donut.axis("off")

    plt.tight_layout()
    plt.savefig("circle_cum.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
# Figure 2 : No degrowth avec cuts et température
fuel_categories = ["E-Fuel", "Biofuel", "LCAF", "Kerosene"]
fuel_keys = ["saf_co2", "saf_ftg", "lcaf", "kerosene"]

# Années de contrôle pour les ticks
control_years = [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060, 2065, 2070]

# Mapping des labels pour cell 11
label_mapping_cell11 = {
    "slsqp_co2_no_degrowth.json": r"RU + NRD - Obj: Cum. $CO_2$ (2070)",
    "slsqp_co2_no_degrowth_cut_2050.json": r"RU + NRD - Obj: Cum. $CO_2$ (2050)",
    "slsqp_co2_no_degrowth_cut_2060.json": r"RU + NRD - Obj: Cum. $CO_2$ (2060)",
    "slsqp_end_temp_no_degrowth_start2.json": r"RU + NRD - Obj. T°C 2070",
    "slsqp_end_temp_cut_2050.json": r"RU + NRD - Obj. T°C 2050",
    "slsqp_end_temp_cut_2060.json": r"RU + NRD - Obj. T°C 2060",
}

# Créer une figure avec 4 subplots (1 par catégorie)
fig_no_deg, axes_no_deg = plt.subplots(2, 2, figsize=(18, 12))
axes_no_deg = axes_no_deg.flatten()

# Filtrer les résultats pour la référence et les no_degrowth
files_to_include = [
    "is2medium_outputs.json",
    "slsqp_co2_no_degrowth.json",
    "slsqp_co2_no_degrowth_cut_2050.json",
    "slsqp_co2_no_degrowth_cut_2060.json",
    "slsqp_end_temp_no_degrowth_start2.json",
    "slsqp_end_temp_cut_2050.json",
    "slsqp_end_temp_cut_2060.json",
]
filtered_results = [item for item in results if any(f in item["file"] for f in files_to_include)]

# Palette de couleurs
n_files = len(filtered_results)
file_colors = plt.cm.tab10(np.linspace(0, 1, n_files))

# Markers variés
markers = ["o", "s", "^", "v", "D", "p", "*", "h", "x", "+"]

for fuel_idx, (fuel_name, fuel_key) in enumerate(zip(fuel_categories, fuel_keys)):
    ax = axes_no_deg[fuel_idx]

    # Tracer une ligne pour chaque fichier JSON
    for file_idx, item in enumerate(filtered_results):
        df = item["data"]
        years = BASE_YEAR + np.arange(len(df))
        if len(years) == 0:
            continue

        filename = item["file"]
        cut_year = item.get("cut_year")

        # Cas spécial pour la référence
        if "is2medium_outputs" in filename:
            label = "IS2 reference"
            color = "#404040"  # Gris foncé
            linewidth = 2.5
            linestyle = "-"
            marker = None
            markersize = 0
            markevery = None
            alpha = 1.0
        else:
            label = label_mapping_cell11.get(filename, filename.replace(".json", ""))
            color = file_colors[file_idx]
            linewidth = 2

            # Line style selon le type d'objectif
            if "temperature" in filename.lower():
                linestyle = "--"  # Pointillés pour température
            else:
                linestyle = "-"  # Ligne continue pour no_degrowth

            # Ajouter des marqueurs réguliers pour toutes les courbes
            marker = markers[file_idx % len(markers)]
            markersize = 5
            # Marqueurs aux années de contrôle
            mark_indices = [i for i, year in enumerate(years) if year in control_years]
            markevery = mark_indices if mark_indices else None

            alpha = 0.7

        ax.plot(
            years,
            df[fuel_key],
            color=color,
            label=label,
            linewidth=linewidth,
            linestyle=linestyle,
            alpha=alpha,
            marker=marker,
            markersize=markersize,
            markevery=markevery,
            markeredgewidth=1.5,
            markerfacecolor="none",
            markeredgecolor=color,
        )

        # Marqueur accentué au cut
        if cut_year is not None:
            cut_idx = np.where(years == cut_year)[0]
            if len(cut_idx) > 0:
                cx = years[cut_idx[0]]
                cy = df[fuel_key].iloc[cut_idx[0]]
                ax.plot(
                    cx,
                    cy,
                    marker=marker if marker is not None else markers[file_idx % len(markers)],
                    markersize=max(markersize, 11),
                    markeredgewidth=2.5,
                    markerfacecolor="none",
                    markeredgecolor=color,
                    linestyle="None",
                    alpha=1.0,
                )

        # Marqueur accentué à la fin de chaque tracé (taille variable)
        if len(years) > 0 and "is2medium_outputs" not in filename:
            last_x = years[-1]
            last_y = df[fuel_key].iloc[-1]
            # Taille croissante pour chaque fichier pour éviter superposition
            end_markersize = 11 + file_idx * 2
            ax.plot(
                last_x,
                last_y,
                marker=marker if marker is not None else markers[file_idx % len(markers)],
                markersize=end_markersize,
                markeredgewidth=2.5,
                markerfacecolor="none",
                markeredgecolor=color,
                linestyle="None",
                alpha=1.0,
            )

    ax.set_ylabel("Share [%]", fontsize=12)
    ax.set_xlabel("Year", fontsize=12)
    ax.set_title(f"{fuel_name} share", fontsize=14, fontweight="bold")
    ax.set_ylim(0, 100)
    ax.set_xlim(PLOT_START_YEAR, PLOT_END_YEAR)
    tick_years = [y for y in control_years if y <= PLOT_END_YEAR]
    ax.set_xticks(tick_years)
    ax.set_xticklabels([str(y) for y in tick_years], rotation=45)
    if fuel_idx == 2:
        ax.legend(loc="best", fontsize=15, ncol=1, framealpha=0.9)
    ax.grid(True, alpha=0.3, which="both")
    ax.grid(True, alpha=0.5, which="major", linewidth=0.8)

plt.tight_layout()
plt.savefig("no_degrowth_scenarios.pdf", bbox_inches="tight")

plt.show()